# Intelligent News Categorization System
## Phase 1: Project Setup, Dataset Exploration & Integration

### Objective
Build a unified, robust, 5-class news dataset to classify headlines and full articles into:
1. **Politics**
2. **Business**
3. **Technology**
4. **Sports**
5. **Entertainment**

### Key Constraints & Rules
- **No Transformers / LLMs**: Only classical ML (Logistic Regression, SVM) and Deep Learning (RNN, LSTM) in future phases.
- **No Data Leakage**: Do not use test splits for fitting vectorizers or vocabularies.
- **Verified Public Datasets**: BBC News Full-Text (Greene & Cunningham 2006) + HuffPost News Category Dataset (Misra 2022).
- **Strict Category Mapping**: General world news is strictly excluded to prevent mislabeling as Politics.

### Step 1: Colab Environment Setup & Dependencies
Install necessary libraries and configure directories.

In [ ]:
# Install dependencies (datasets library for quick raw data fetching)
!pip install -q pandas numpy scikit-learn matplotlib seaborn datasets

import os
import re
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional Google Drive mount
try:
    from google.colab import drive
    # Uncomment below if you want to persist data to your Drive:
    # drive.mount('/content/drive')
    print('Google Colab environment detected.')
except ImportError:
    print('Local or standard Python environment detected.')

# Create directories
os.makedirs('dataset/raw', exist_ok=True)
os.makedirs('dataset/processed', exist_ok=True)
os.makedirs('evaluation', exist_ok=True)
print('Project directories initialized.')

### Step 2: Source 1 - BBC News Classification Dataset
- **Origin**: Derek Greene & Padraig Cunningham, ICML 2006 (UCD Insight Centre).
- **Content**: 2,225 full-text articles across 5 classes: business, entertainment, politics, sport, tech.
- **License**: Non-commercial academic research use.

In [ ]:
bbc_url = 'http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip'
bbc_zip = 'dataset/raw/bbc-fulltext.zip'

if not os.path.exists(bbc_zip):
    print('Downloading BBC News dataset from UCD...')
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(bbc_url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(bbc_zip, 'wb') as f:
        f.write(resp.read())
    print('Download complete.')

# Extract
with zipfile.ZipFile(bbc_zip, 'r') as zip_ref:
    zip_ref.extractall('dataset/raw')

bbc_map = {
    'business': 'Business',
    'entertainment': 'Entertainment',
    'politics': 'Politics',
    'sport': 'Sports',
    'tech': 'Technology'
}

records_bbc = []
base_path = 'dataset/raw/bbc'
for folder, cat_name in bbc_map.items():
    folder_p = os.path.join(base_path, folder)
    if os.path.isdir(folder_p):
        for fname in sorted(os.listdir(folder_p)):
            if fname.endswith('.txt'):
                with open(os.path.join(folder_p, fname), 'r', encoding='latin-1') as f:
                    txt = f.read().strip()
                    if txt:
                        records_bbc.append({
                            'text': txt,
                            'category': cat_name,
                            'source': 'BBC News',
                            'text_type': 'full_article'
                        })

df_bbc = pd.DataFrame(records_bbc)
print(f'BBC News loaded: {len(df_bbc)} articles.')
print(df_bbc['category'].value_counts())

### Step 3: Source 2 - HuffPost News Category Dataset
- **Author**: Rishabh Misra (arXiv:2205.10512, 2022).
- **License**: CC BY 4.0.
- **Content**: Headline and short summaries.
- **Mapping**: POLITICS -> Politics, BUSINESS -> Business, TECH -> Technology, SPORTS -> Sports, ENTERTAINMENT -> Entertainment.
- **Explicit Exclusion**: WORLD NEWS and other categories are strictly dropped.

In [ ]:
from datasets import load_dataset

print('Loading HuffPost dataset from hub...')
ds_huff = load_dataset('heegyu/news-category-dataset', split='train')

huff_map = {
    'POLITICS': 'Politics',
    'BUSINESS': 'Business',
    'TECH': 'Technology',
    'SPORTS': 'Sports',
    'ENTERTAINMENT': 'Entertainment'
}

records_huff = []
for row in ds_huff:
    cat = row.get('category', '')
    if cat in huff_map:
        headline = (row.get('headline') or '').strip()
        desc = (row.get('short_description') or '').strip()
        if headline and desc:
            txt = f'{headline} - {desc}'
        elif headline:
            txt = headline
        elif desc:
            txt = desc
        else:
            continue
        records_huff.append({
            'text': txt,
            'category': huff_map[cat],
            'source': 'HuffPost',
            'text_type': 'headline_summary'
        })

df_huff = pd.DataFrame(records_huff)
print(f'Total matching HuffPost records: {len(df_huff)}')
print(df_huff['category'].value_counts())

# Stratified sample 2,000 per class to keep classes balanced
balanced_huff = df_huff.groupby('category', group_keys=False).apply(
    lambda g: g.sample(n=min(2000, len(g)), random_state=42)
).reset_index(drop=True)
print(f'Balanced HuffPost sample: {len(balanced_huff)} records.')

### Step 4: Integration, Cleaning, and Deduplication
Concatenate datasets, clean whitespaces, eliminate missing values, and remove exact and near-duplicates.

In [ ]:
# Merge
df_unified = pd.concat([df_bbc, balanced_huff], ignore_index=True)
initial_count = len(df_unified)

# Normalize whitespaces and clean text
df_unified['text'] = df_unified['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df_unified = df_unified[df_unified['text'].str.len() > 15].copy()

# Exact deduplication
df_unified = df_unified.drop_duplicates(subset=['text']).reset_index(drop=True)
exact_dupes = initial_count - len(df_unified)

# Near-deduplication (case-insensitive alphanumeric normalization)
norm_series = df_unified['text'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True)
near_dupes = norm_series.duplicated().sum()
df_unified = df_unified[~norm_series.duplicated()].reset_index(drop=True)

# Compute text length statistics
df_unified['word_count'] = df_unified['text'].apply(lambda x: len(x.split()))
df_unified['char_count'] = df_unified['text'].apply(len)

print(f'Initial rows: {initial_count}')
print(f'Exact duplicates removed: {exact_dupes}')
print(f'Near duplicates removed: {near_dupes}')
print(f'Cleaned dataset total rows: {len(df_unified)}')

### Step 5: Dataset Analysis & Visualizations
Evaluate class distributions and text length characteristics.

In [ ]:
categories = ['Politics', 'Business', 'Technology', 'Sports', 'Entertainment']

# 1. Class Distribution
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
ax = sns.countplot(data=df_unified, x='category', order=categories, palette='viridis')
plt.title('Overall Class Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Articles')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.subplot(1, 2, 2)
sns.countplot(data=df_unified, x='category', hue='source', order=categories, palette='Set2')
plt.title('Distribution by Source', fontsize=12, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Articles')
plt.tight_layout()
plt.savefig('evaluation/class_distribution.png', dpi=300)
plt.show()

# 2. Text Length Distribution
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(data=df_unified[df_unified['word_count'] < 1000], x='word_count', hue='source', kde=True, bins=35)
plt.title('Word Count Distribution (Truncated at 1000 words)', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
sns.boxplot(data=df_unified, x='category', y='word_count', showfliers=False, order=categories, palette='pastel')
plt.title('Word Count Across Categories', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation/text_length_distribution.png', dpi=300)
plt.show()

### Step 6: Sample Articles from Every Category
Verify text quality and topical authenticity across classes.

In [ ]:
for cat in categories:
    print('=' * 80)
    print(f'CATEGORY: {cat.upper()}')
    samples = df_unified[df_unified['category'] == cat].sample(2, random_state=42)
    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        snippet = row['text'][:220] + ('...' if len(row['text']) > 220 else '')
        print(f'  Sample {idx} [{row["source"]}] ({row["word_count"]} words):')
        print(f'    "{snippet}"\n')

### Step 7: Export Unified Cleaned CSV
Save `news_dataset_cleaned.csv` to `dataset/processed/`.

In [ ]:
out_path = 'dataset/processed/news_dataset_cleaned.csv'
cols = ['text', 'category', 'source', 'text_type', 'word_count', 'char_count']
df_unified[cols].to_csv(out_path, index=False)
print(f'Successfully exported clean dataset to: {out_path}')
print(f'Dataset shape: {df_unified.shape}')